# Day 4 — ILT 2: Ingestion Patterns Recap — All 4 Sources
### GlobalMart Data Engineering · 12:00 PM – 1:00 PM

---

## Session Objectives

By the end of this session you will be able to:
- Describe the ingestion pattern for each of GlobalMart's 4 data sources
- Compare batch vs streaming ingestion trade-offs across sources
- Explain how each source lands in the Bronze layer
- Identify the correct trigger mode (availableNow vs processingTime) per source
- Articulate the end-to-end flow from raw source to Bronze Delta table

---

## Agenda

| Time | Topic |
|------|-------|
| 12:00 | GlobalMart architecture recap |
| 12:10 | Source 1 — Supabase (Postgres via CDC/JDBC) |
| 12:20 | Source 2 — Public APIs (REST/HTTP) |
| 12:30 | Source 3 — Neo4j (Graph Database) |
| 12:40 | Source 4 — ADLS Flat Files (Auto Loader) |
| 12:50 | Cross-source comparison + what's next |
| 12:55 | Q&A |

---
## GlobalMart Architecture — Quick Recap

```
┌─────────────────────────────────────────────────────────────────┐
│                    GLOBALMART DATA SOURCES                       │
│                                                                   │
│  ┌──────────────┐  ┌──────────────┐  ┌───────────┐  ┌────────┐  │
│  │  Supabase    │  │  Public APIs │  │   Neo4j   │  │  ADLS  │  │
│  │  (Postgres)  │  │  (REST/HTTP) │  │ (GraphDB) │  │  Files │  │
│  │              │  │              │  │           │  │        │  │
│  │  Orders      │  │  Weather     │  │  Supplier │  │  Flat  │  │
│  │  Customers   │  │  Promotions  │  │  Networks │  │  CSVs  │  │
│  │  Payments    │  │  Exchange    │  │  Routes   │  │        │  │
│  └──────┬───────┘  └──────┬───────┘  └─────┬─────┘  └───┬────┘  │
└─────────┼─────────────────┼────────────────┼────────────┼───────┘
          │                 │                │            │
          ▼                 ▼                ▼            ▼
┌─────────────────────────────────────────────────────────────────┐
│                    BRONZE LAYER (Delta Lake)                      │
│   bronze_orders  bronze_customers  bronze_api  bronze_graph       │
└─────────────────────────────────────────────────────────────────┘
          │
          ▼
       SILVER → GOLD → Reporting / Genie
```

### Why 4 Sources?

Real enterprise systems never have a single source of truth. GlobalMart mirrors reality:

| Source | Why It Exists |
|--------|---------------|
| Supabase (Postgres) | Transactional OLTP system — orders, customers, payments |
| Public APIs | External data enrichment — weather affects delivery, FX rates affect pricing |
| Neo4j | Relationship data — supplier networks, delivery routes, product graphs |
| ADLS Flat Files | Supplier EDI feeds, bulk data drops, legacy system exports |

---
## Source 1 — Supabase (PostgreSQL)

### What Is It?
Supabase is a hosted PostgreSQL database. GlobalMart's core transactional system — every order, customer, and payment lives here.

### The Ingestion Challenge
```
Problem: Postgres is an OLTP database — optimised for fast writes, not bulk reads.
         Querying the full orders table (millions of rows) every hour kills performance.
         We need CHANGES ONLY — what was inserted/updated/deleted since last run.
```

### Pattern 1 — CDC via WAL (Preferred)

```
Postgres WAL (Write-Ahead Log)
      |
      ▼  logical replication slot
Lakeflow Connect (Databricks)   ← or Debezium + Kafka
      |
      ▼  INSERT / UPDATE / DELETE events
Bronze Delta Table
      |
      ▼  MERGE (upsert)
Silver orders_clean
```

**Key concepts:**
- WAL = append-only log of every database change
- Logical replication slot = a named cursor in the WAL so we don't miss events
- Lakeflow Connect reads the slot, converts CDC events to Delta rows
- Each row has: `_op` (INSERT/UPDATE/DELETE), `_ts_ms` (event timestamp), full row data

**Trigger mode:** `processingTime` (continuous) or `availableNow` (scheduled batch)

---

### Pattern 2 — JDBC Batch (Simple but Costly)

```python
# Pull only rows changed since last watermark
last_watermark = get_watermark()   # from control table

df = (
    spark.read
    .format("jdbc")
    .option("url",      "jdbc:postgresql://host/db")
    .option("dbtable",  f"(SELECT * FROM orders WHERE updated_at > '{last_watermark}') AS q")
    .option("user",     "user")
    .option("password", "YOUR_SUPABASE_PASSWORD")  # never hardcode
    .load()
)

# Write to Bronze
df.write.format("delta").mode("append").save(bronze_orders_path)

# Update watermark
save_watermark(df.agg({"updated_at": "max"}).collect()[0][0])
```

**Trade-off vs CDC:**

| | CDC (WAL) | JDBC Batch |
|--|-----------|------------|
| Captures deletes? | Yes | No (hard to detect) |
| Source load | Low (reads WAL) | Medium (queries table) |
| Complexity | Higher | Lower |
| Latency | Near real-time | Depends on schedule |
| What we use | Preferred | Fallback |

---
## Source 2 — Public APIs (REST/HTTP)

### What Is It?
External REST APIs that provide data enrichment for GlobalMart:
- Weather API → correlates weather with delivery delays
- Exchange rate API → converts international order values to INR
- Promotions API → active discount codes and campaign data

### The Ingestion Challenge
```
Problem: APIs are rate-limited (e.g. 100 calls/minute).
         Responses are paginated (can't get all data in one call).
         No streaming option — must poll on a schedule.
         Response schema can change with API version updates.
```

### Pattern — Scheduled HTTP Pull

```python
import requests

def fetch_all_pages(base_url, headers, params):
    results = []
    page    = 1
    while True:
        resp = requests.get(base_url, headers=headers, params={**params, "page": page})
        data = resp.json()
        if not data.get("results"):   # no more pages
            break
        results.extend(data["results"])
        page += 1
    return results

# Fetch and convert to Spark DataFrame
raw_data = fetch_all_pages(
    base_url = "https://api.example.com/promotions",
    headers  = {"Authorization": "Bearer YOUR_API_KEY"},
    params   = {"date": "2026-06-15", "per_page": 100}
)

df = spark.createDataFrame(raw_data)
df = df.withColumn("_ingested_at", current_timestamp())
       .withColumn("_source", lit("promotions_api"))

df.write.format("delta").mode("append").save(bronze_api_path)
```

### Key Considerations

| Concern | Handling |
|---------|----------|
| Rate limiting | Sleep between pages, use retry with backoff |
| Pagination | Loop until empty page, use cursor/offset |
| Schema changes | `mergeSchema=true` on Delta write |
| API downtime | Checkpoint last successful run; skip+alert on failure |
| PII in responses | Mask/hash before writing to Bronze |

**Trigger mode:** Batch (`spark.read` / manual schedule via Databricks Workflows)

> APIs don't push data to you — you must pull on a schedule. Databricks Workflows handles scheduling (covered in Week 3).

---
## Source 3 — Neo4j (Graph Database)

### What Is It?
Neo4j stores relationship data as a graph of nodes and edges:
- **Nodes:** Suppliers, Products, Warehouses, Delivery Zones
- **Edges:** `SUPPLIES`, `SHIPS_TO`, `LOCATED_IN`, `COMPETES_WITH`

Question a graph answers that SQL cannot easily:
> *"Find all suppliers who supply products in Category X, who ship to Zone Y, and who have at least 3 hops to our primary warehouse"*

### The Ingestion Challenge
```
Problem: Graph data is not tabular — it's nodes and relationships.
         Must be flattened into rows before landing in Delta.
         Cypher (Neo4j's query language) is needed to extract data.
         Volume is typically lower than Postgres, but relationships are complex.
```

### Pattern — Cypher Query via Spark Connector

```python
# Using the Neo4j Connector for Apache Spark
df_suppliers = (
    spark.read
    .format("org.neo4j.spark.DataSource")
    .option("url",      "bolt://your-neo4j-host:7687")
    .option("authentication.basic.username", "neo4j")
    .option("authentication.basic.password", "YOUR_NEO4J_PASSWORD")
    .option("query", """
        MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)
        RETURN s.supplierID   AS SupplierID,
               s.name         AS SupplierName,
               p.productID    AS ProductID,
               p.category     AS Category
    """)
    .load()
)

df_suppliers \
    .withColumn("_ingested_at", current_timestamp()) \
    .write.format("delta").mode("overwrite").save(bronze_graph_suppliers_path)
```

### Why `overwrite` for Graph Data?

```
Graph data = relationship snapshot (not transactional events)

Every time you query the graph, you get the CURRENT state of all relationships.
There's no WAL, no CDC, no delta — just the current graph.

Strategy:
  - Full overwrite daily (or per schedule)
  - Use Delta time travel if you need historical graph snapshots
  - Compare today's graph vs yesterday's via time travel to detect relationship changes
```

### Cypher vs SQL — Quick Comparison

```cypher
-- Cypher: find all suppliers 2 hops from warehouse W1
MATCH (w:Warehouse {id: 'W1'})-[:CONNECTS*1..2]-(s:Supplier)
RETURN DISTINCT s.supplierID, s.name
```

```sql
-- SQL equivalent: requires self-joins or recursive CTEs — very verbose
WITH RECURSIVE hops AS (
    SELECT supplier_id, 1 AS depth FROM routes WHERE warehouse_id = 'W1'
    UNION ALL
    SELECT r.supplier_id, h.depth + 1
    FROM routes r JOIN hops h ON r.warehouse_id = h.supplier_id
    WHERE h.depth < 2
)
SELECT DISTINCT supplier_id FROM hops;
```

**Trigger mode:** Batch (`spark.read`), scheduled daily or weekly

---
## Source 4 — ADLS Flat Files (Auto Loader)

### What Is It?
Suppliers send bulk CSV/JSON/Parquet files directly to ADLS Gen2:
- `orders.csv` — bulk order exports from partner systems
- `shipping_tier.csv` — supplier shipping rates
- `products.csv` — product catalogue updates

### The Ingestion Challenge
```
Problem: Files land at unpredictable times.
         Must process each file EXACTLY ONCE — no re-reads, no skips.
         Schema can evolve (new columns added by supplier).
         File volume varies — 5 rows in orders_new.csv, 1.4M rows in orders.csv.
```

### Pattern — Auto Loader (cloudFiles)

```python
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",      "true")
    .option("inferSchema", "true")
    .load(raw_path)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

query = (
    raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema",        "true")
    .trigger(processingTime="30 seconds")
    .start(output_path)
)
```

### What Makes Auto Loader Special

```
Without Auto Loader (spark.read loop):
  Run 1: reads orders.csv → 1.4M rows
  Run 2: reads orders.csv AGAIN → 1.4M rows DUPLICATE + 5 new rows
  Problem: 1.4M duplicates every run

With Auto Loader (cloudFiles):
  Run 1: reads orders.csv → 1.4M rows → checkpoint records orders.csv
  Run 2: sees orders.csv in checkpoint → SKIP
          sees orders_new.csv → NEW → 5 rows processed
  Result: zero duplicates, exactly-once delivery
```

**Trigger modes:**
- `availableNow=True` → process all pending files, stop (scheduled batch style)
- `processingTime="30 seconds"` → continuous, detect files as they land

> This is exactly what we built and demonstrated in the Auto Loader DEMO notebook.

---
## Cross-Source Comparison

| Dimension | Supabase (Postgres) | Public APIs | Neo4j (Graph) | ADLS Files |
|-----------|--------------------|-----------|--------------|-----------|
| **Data type** | Transactional rows | JSON payloads | Nodes + edges | CSV/JSON/Parquet |
| **Ingestion method** | CDC (WAL) / JDBC | HTTP pull | Cypher + Spark connector | Auto Loader (cloudFiles) |
| **Trigger mode** | Continuous / scheduled | Scheduled batch | Scheduled batch | Continuous or batch |
| **Exactly-once?** | Yes (Lakeflow/WAL offset) | Idempotent re-fetch | Full overwrite | Yes (checkpoint) |
| **Volume** | High (millions of rows) | Low-medium | Low-medium | High (bulk files) |
| **Schema evolution** | Managed by Postgres DDL | API versioning | Graph schema flexible | Auto Loader addNewColumns |
| **Latency** | Near real-time (CDC) | Minutes (poll interval) | Hours (daily batch) | Seconds to minutes |
| **Bronze write mode** | Append (CDC events) | Append | Overwrite (snapshot) | Append |

---

## Bronze Landing Strategy — Per Source

```
bronze/
  ├── orders_cdc/          ← Supabase CDC events (append)
  │     _op, _ts_ms, OrderID, CustomerID, ...
  │
  ├── api_promotions/      ← Promotions API (append daily)
  │     promo_id, discount_pct, start_date, end_date, _ingested_at
  │
  ├── graph_supplier_network/  ← Neo4j snapshot (overwrite daily)
  │     SupplierID, SupplierName, ProductID, Category, _ingested_at
  │
  └── flat_files/          ← Auto Loader from ADLS raw/ (append)
        OrderID, CustomerID, ..., _ingested_at, _source_file
```

**Bronze rules (same for all sources):**
1. Never transform — land raw data as close to source as possible
2. Always add `_ingested_at` and `_source` audit columns
3. Never delete — Bronze is append-only (except overwrite snapshots like Neo4j)
4. Partition by ingestion date where volume warrants it

---
## What Comes Next — The Silver Layer

Bronze is raw, messy, and multi-schema. Silver is where the real engineering happens:

```
Bronze (raw)                Silver (clean)              Gold (business-ready)
─────────────────────       ─────────────────────       ─────────────────────
ordersCSV (flat)      →     orders_clean          →     fact_orders
cdc_events (INSERT/   →     customers_current     →     dim_customer
  UPDATE/DELETE)            (latest state via MERGE)     (SCD Type 2)
api_promotions        →     promotions_active     →     dim_promotions
graph_suppliers       →     suppliers_network     →     supplier_performance
```

**Silver transformations (Week 2):**
- Cast data types explicitly (string dates → timestamp)
- Standardise column names (CamelCase, snake_case)
- Apply MERGE for CDC events (upsert to latest state)
- Deduplicate (window functions on OrderID)
- Enrich (join flat files to API data)
- Apply data quality constraints (DLT expectations or manual checks)

---
## Summary — Ingestion Patterns at a Glance

```
4 Sources → 4 Patterns → 1 Bronze Layer

Supabase  ──── CDC (WAL) ──────────────────────► bronze/orders_cdc/
               ↑ continuous, exactly-once
               ↑ captures inserts + updates + deletes

APIs      ──── HTTP Pull (scheduled) ───────────► bronze/api_*/
               ↑ batch, idempotent
               ↑ pagination + rate limit handling

Neo4j     ──── Cypher + Spark connector ─────────► bronze/graph_*/
               ↑ batch, full overwrite daily
               ↑ flattened from graph → rows

ADLS      ──── Auto Loader (cloudFiles) ─────────► bronze/flat_files/
               ↑ continuous or batch
               ↑ exactly-once via checkpoint
               ↑ schema evolution via addNewColumns
```

---

## Key Takeaways

1. **No single pattern fits all sources** — match the ingestion method to the source's nature
2. **CDC > JDBC** for transactional sources — captures deletes, lower source load
3. **Auto Loader > spark.read loop** for files — exactly-once, schema evolution built in
4. **APIs and Graphs = batch** — they don't support streaming push; you must pull
5. **Bronze = raw + audit columns** — the ingestion layer, never the transformation layer
6. **All 4 sources land in Bronze as Delta tables** — same format, same guarantees

---

## Discussion Questions

1. *Why can't we use Auto Loader for the Supabase Postgres source?*

2. *A new supplier sends files every 5 minutes. Should we use `availableNow=True` or `processingTime`? Why?*

3. *The Neo4j graph is overwritten daily. How would you use Delta time travel to compare supplier relationships between today and last week?*

4. *An API starts returning a new field `delivery_score` in its response. What happens in the Bronze write? What config handles this automatically?*

5. *Why does Bronze use append mode for CDC events rather than upsert (MERGE)?*